# Bern Stipendium - Interactive Form

Showcase notebook for the OpenFisca implementation of the Bernese **Gesetz über die Ausbildungsbeiträge (ABG)** and the **Verordnung (ABV)** in [`src/bern_stipendium`](../src/bern_stipendium/).

This notebook builds an `ipywidgets` form that lets you assemble a scholarship application from scratch and computes the resulting **Stipendium** and **Darlehen** amounts plus the full eligibility breakdown.

Sections:
1. Bootstrap the tax-benefit system
2. Helper utilities (situation builder, result formatter)
3. Build the interactive form (applicant, parents, partner, household)
4. Compute & display the result
5. Preset scenarios for quick demos
6. Single-input sensitivity sweep

## 1. Bootstrap

In [12]:
import sys, logging
from pathlib import Path

REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Suppress the harmless package-metadata warning that openfisca-core emits
# when the package isn't pip-installed.
logging.raiseExceptions = False

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import matplotlib.pyplot as plt
from openfisca_core.simulation_builder import SimulationBuilder

from bern_stipendium import CountryTaxBenefitSystem

tbs = CountryTaxBenefitSystem()
PERIOD = "2024"
print(f"Loaded {len(tbs.variables)} variables for period {PERIOD}.")

Loaded 89 variables for period 2024.


## 2. Helpers

`build_situation(...)` translates the form values into the OpenFisca situation dict (with `{period: value}` wrapping). `compute(...)` runs the simulation and returns a dict of named outputs. `render_results(...)` formats the result as HTML.

In [3]:
PERSON_FIELD_NAMES = {
    # Applicant-only inputs
    "wohnsitz_grundlage", "staatsangehoerigkeit", "ausbildungstyp",
    "ausbildungsstufe", "ausbildungsstaette_anerkannt",
    "private_ausbildungsstaette", "private_ausbildungsstaette_qualitaetsgesichert",
    "in_ausbildung", "aktuelles_ausbildungsjahr", "kumulierte_ausbildungsjahre",
    "wohnform", "zivilstand", "alter",
    "beruflicher_wiedereinstieg_nach_familienphase",
    "wichtige_gruende_altersausnahme",
    "jahre_vollerwerbstaetigkeit",
    "pendelzeit_zu_eltern_minuten",
    "andere_zwingende_gruende_eigener_haushalt",
    "hat_eigene_kinder",
    "jahre_faktische_partnerschaft",
    "mahlzeiten_auswaerts_ausbildung_pro_jahr",
    "fahrkosten_ausbildung",
    "kinderbetreuungskosten",
    "tatsaechliche_ausbildungskosten",
    "kumulierte_darlehen",
    # Per-Person financial inputs (apply to applicant, parents, partner)
    "total_einkuenfte_steuerveranlagung",
    "steuerbares_vermoegen",
    "erwerbseinkommen",
    "ergaenzungsleistungen",
    "eigenmietwert",
    "unterhaltsleistung_an_auszubildenden",
    "bvg_beitraege_selbststaendig",
    "saeule_3a_ueberschiessend",
    "selbststaendig",
    "bezahlte_steuern",
    "fahrkosten_arbeit",
    "auswaertige_verpflegung_arbeit",
    "krankenkasse_praemienverbilligung",
}

HOUSEHOLD_FIELD_NAMES = {
    "parent_a_zahlt_unterhalt", "parent_b_zahlt_unterhalt",
    "wohnkosten_familienbudget_tatsaechlich",
    "wohnkosten_persoenlich_tatsaechlich",
}


def _wrap(person_dict, period=PERIOD):
    """Wrap each scalar value as {period: value}."""
    return {k: {period: v} for k, v in person_dict.items() if v is not None}


def build_situation(applicant, parent_a=None, parent_b=None, partner=None,
                     kinder_im_haushalt=0, household=None, period=PERIOD):
    persons = {"applicant": _wrap(applicant, period)}
    roles = {"auszubildende": ["applicant"]}
    if parent_a:
        persons["parent_a"] = _wrap(parent_a, period)
        roles["parents_a"] = ["parent_a"]
    if parent_b:
        persons["parent_b"] = _wrap(parent_b, period)
        roles["parents_b"] = ["parent_b"]
    if partner:
        persons["partner"] = _wrap(partner, period)
        roles["partners"] = ["partner"]
    if kinder_im_haushalt:
        kid_ids = [f"kind_{i+1}" for i in range(kinder_im_haushalt)]
        for kid in kid_ids:
            persons[kid] = {"alter": {period: 10}}
        roles["kinder_im_haushalt"] = kid_ids
    hh = {"h1": dict(roles)}
    if household:
        hh["h1"].update(_wrap(household, period))
    return {"persons": persons, "households": hh}


OUTPUT_GROUPS = [
    ("Eligibility (booleans)", [
        "hat_stipendienrechtlichen_wohnsitz_bern",
        "erfuellt_persoenlichen_status",
        "ausbildungstyp_anerkannt",
        "ausbildungsstaette_qualifiziert",
        "stipendium_grundsaetzlich_ausgeschlossen",
        "verzicht_auf_anrechnung_eltern",
        "beduerftig",
        "maximale_beitragsdauer_eingehalten",
        "altersgrenze_eingehalten",
        "eigener_haushalt_anerkannt",
        "partner_qualifiziert",
        "stipendium_anspruch",
        "darlehen_anspruch",
    ]),
    ("Familienbudget (Art. 13-24 ABV)", [
        "haushaltsgroesse",
        "familienbudget_einkommen",
        "familienbudget_vermoegensanrechnung",
        "familienbudget_grundbedarf",
        "familienbudget_wohnkosten",
        "familienbudget_krankenkasse",
        "familienbudget_integrationszulage",
        "familienbudget_einkommensfreibetrag",
        "familienbudget_freibetraege_total",
        "familienbudget_situationsbedingte_kosten",
        "familienbudget_lebenshaltungskosten",
        "familienbudget_saldo",
    ]),
    ("Persoenliches Budget (Art. 25-33 ABV)", [
        "persoenlich_personen_anzahl",
        "persoenlich_einkommen",
        "persoenlich_vermoegensanrechnung",
        "persoenlich_grundbedarf",
        "persoenlich_wohnkosten",
        "persoenlich_krankenkasse",
        "persoenlich_ausbildungskosten",
        "persoenlich_situationsbedingte_kosten",
        "familienbudget_ueberschuss_anteil_persoenlich",
        "familienbudget_fehlbetrag_pro_kopf_anteil",
    ]),
    ("Final amounts", [
        "anerkannte_ausbildungskosten",
        "anrechenbare_mittel_total",
        "fehlbetrag",
        "stipendium_quote",
        "stipendium_betrag",
        "darlehen_betrag",
    ]),
]


def compute(situation, person_id="applicant", period=PERIOD):
    sb = SimulationBuilder()
    sim = sb.build_from_dict(tbs, situation)
    person_idx = list(situation["persons"].keys()).index(person_id)
    results = {}
    for _, names in OUTPUT_GROUPS:
        for var in names:
            entity = tbs.variables[var].entity
            arr = sim.calculate(var, period)
            if entity.key == "person":
                results[var] = arr[person_idx]
            else:
                results[var] = arr[0]
    return results, sim


def _fmt(value):
    if isinstance(value, (bool, )) or str(type(value)) == "<class 'numpy.bool_'>":
        return "YES" if bool(value) else "NO"
    if isinstance(value, float) or str(type(value)) == "<class 'numpy.float32'>" or str(type(value)) == "<class 'numpy.float64'>":
        v = float(value)
        if v == int(v):
            return f"{int(v):,}"
        return f"{v:,.2f}"
    return str(value)


def render_results(results):
    sections = []
    for title, names in OUTPUT_GROUPS:
        rows = "".join(
            f"<tr><td style='padding:2px 12px 2px 0'>{var}</td>"
            f"<td style='padding:2px 0;text-align:right;font-family:monospace'>{_fmt(results[var])}</td></tr>"
            for var in names
        )
        sections.append(
            f"<h4 style='margin-bottom:4px'>{title}</h4>"
            f"<table style='border-collapse:collapse'>{rows}</table>"
        )
    headline = (
        f"<div style='padding:10px;background:#f0f0f0;border-radius:6px;margin-bottom:12px'>"
        f"<b>Stipendium:</b> CHF {_fmt(results['stipendium_betrag'])} &nbsp;|&nbsp; "
        f"<b>Darlehen:</b> CHF {_fmt(results['darlehen_betrag'])} &nbsp;|&nbsp; "
        f"<b>Stipendium-Anspruch:</b> {_fmt(results['stipendium_anspruch'])}"
        f"</div>"
    )
    return HTML(headline + "<br>".join(sections))

## 3. The form

Each tab below collects one chunk of the situation. Defaults are set so that running the form unchanged simulates a 22-year-old apprentice in their own household with no parental support.

In [4]:
# Enum option lists. The dropdown stores the enum *string key* as value (this
# is what OpenFisca expects in the situation dict).
OPT_WOHNSITZ = [
    ("elterlicher_wohnsitz - Eltern wohnen in Bern", "elterlicher_wohnsitz"),
    ("elternlos - eigener Wohnsitz in Bern", "elternlos"),
    ("eltern_im_ausland - eigener Wohnsitz in Bern", "eltern_im_ausland"),
    ("fluechtling_staatenlos", "fluechtling_staatenlos"),
    ("finanziell_unabhaengig", "finanziell_unabhaengig"),
    ("keiner - kein stipendienrechtlicher Wohnsitz", "keiner"),
]
OPT_NATIONALITY = [
    ("schweizer", "schweizer"),
    ("eu_efta_niederlassung", "eu_efta_niederlassung"),
    ("aufenthaltsbewilligung_b", "aufenthaltsbewilligung_b"),
    ("fluechtling_staatenlos", "fluechtling_staatenlos"),
    ("andere - keine Anspruchsberechtigung", "andere"),
]
OPT_AUSB_TYP = [
    ("vorbildung", "vorbildung"),
    ("erstausbildung", "erstausbildung"),
    ("zweitausbildung", "zweitausbildung"),
    ("hoehere_berufsbildung", "hoehere_berufsbildung"),
    ("umschulung", "umschulung"),
    ("primarstufe (nicht anerkannt)", "primarstufe"),
    ("sekundarstufe_i (nicht anerkannt)", "sekundarstufe_i"),
    ("weiterbildung (nicht anerkannt)", "weiterbildung"),
    ("zweite_hochschulausbildung (nicht anerkannt)", "zweite_hochschulausbildung"),
]
OPT_AUSB_STUFE = [
    ("sekundarstufe_ii", "sekundarstufe_ii"),
    ("tertiaerstufe", "tertiaerstufe"),
    ("nicht_anerkannt", "nicht_anerkannt"),
]
OPT_WOHNFORM = [
    ("elterlicher_haushalt", "elterlicher_haushalt"),
    ("eigener_haushalt", "eigener_haushalt"),
    ("gemeinschaftlicher_haushalt - WG/Heim/Internat", "gemeinschaftlicher_haushalt"),
]
OPT_ZIVILSTAND = [
    ("ledig", "ledig"),
    ("verheiratet", "verheiratet"),
    ("eingetragene_partnerschaft", "eingetragene_partnerschaft"),
    ("faktische_partnerschaft", "faktische_partnerschaft"),
    ("geschieden", "geschieden"),
    ("getrennt", "getrennt"),
    ("verwitwet", "verwitwet"),
]

In [5]:
LAYOUT = widgets.Layout(width="360px")
STYLE = {"description_width": "180px"}

# --- Applicant tab ---
w_alter = widgets.IntSlider(value=22, min=15, max=65, description="Alter", style=STYLE, layout=LAYOUT)
w_wohnsitz = widgets.Dropdown(options=OPT_WOHNSITZ, value="finanziell_unabhaengig", description="Wohnsitz Grundl.", style=STYLE, layout=LAYOUT)
w_nationality = widgets.Dropdown(options=OPT_NATIONALITY, value="schweizer", description="Staatsangeh.", style=STYLE, layout=LAYOUT)
w_zivilstand = widgets.Dropdown(options=OPT_ZIVILSTAND, value="ledig", description="Zivilstand", style=STYLE, layout=LAYOUT)
w_wohnform = widgets.Dropdown(options=OPT_WOHNFORM, value="eigener_haushalt", description="Wohnform", style=STYLE, layout=LAYOUT)

w_ausb_typ = widgets.Dropdown(options=OPT_AUSB_TYP, value="erstausbildung", description="Ausbildungstyp", style=STYLE, layout=LAYOUT)
w_ausb_stufe = widgets.Dropdown(options=OPT_AUSB_STUFE, value="sekundarstufe_ii", description="Stufe", style=STYLE, layout=LAYOUT)
w_anerkannt = widgets.Checkbox(value=True, description="Ausbildungsstaette anerkannt", style=STYLE, layout=LAYOUT)
w_privat = widgets.Checkbox(value=False, description="Private Stätte", style=STYLE, layout=LAYOUT)
w_qs = widgets.Checkbox(value=True, description="Privat: QS vorhanden", style=STYLE, layout=LAYOUT)
w_in_ausb = widgets.Checkbox(value=True, description="In Ausbildung", style=STYLE, layout=LAYOUT)
w_jahr = widgets.IntText(value=1, description="Aktuelles Jahr", style=STYLE, layout=LAYOUT)
w_kum_jahre = widgets.IntText(value=0, description="Kum. Beitragsjahre", style=STYLE, layout=LAYOUT)

w_eink = widgets.FloatText(value=0, description="Total Einkuenfte StG", style=STYLE, layout=LAYOUT)
w_verm = widgets.FloatText(value=0, description="Steuerb. Vermoegen", style=STYLE, layout=LAYOUT)
w_erwerb = widgets.FloatText(value=0, description="Erwerbseinkommen", style=STYLE, layout=LAYOUT)
w_ausb_kosten = widgets.FloatText(value=2500, description="Tats. Ausb.kosten", style=STYLE, layout=LAYOUT)
w_kibe = widgets.FloatText(value=0, description="Kinderbetreuung", style=STYLE, layout=LAYOUT)
w_fahrkosten_aus = widgets.FloatText(value=0, description="Fahrkosten Ausb.", style=STYLE, layout=LAYOUT)
w_mahlzeiten = widgets.IntText(value=0, description="Mahlzeiten ausw./J.", style=STYLE, layout=LAYOUT)

w_pendelzeit = widgets.IntText(value=0, description="Pendelzeit (Min.)", style=STYLE, layout=LAYOUT)
w_zwingend = widgets.Checkbox(value=False, description="Andere zwing. Gruende", style=STYLE, layout=LAYOUT)
w_kinder_eigen = widgets.Checkbox(value=False, description="Hat eigene Kinder", style=STYLE, layout=LAYOUT)
w_partnerschaft = widgets.IntText(value=0, description="Faktische Partner. (J.)", style=STYLE, layout=LAYOUT)
w_vef = widgets.IntText(value=0, description="Vollerwerbst. (J.)", style=STYLE, layout=LAYOUT)
w_wiedereinstieg = widgets.Checkbox(value=False, description="Berufl. Wiedereinstieg", style=STYLE, layout=LAYOUT)
w_wichtig = widgets.Checkbox(value=False, description="Wichtige Gruende Alter", style=STYLE, layout=LAYOUT)

w_kum_darlehen = widgets.FloatText(value=0, description="Kum. Darlehen (CHF)", style=STYLE, layout=LAYOUT)

applicant_box = widgets.VBox([
    widgets.HTML("<b>Personal & residency</b>"),
    widgets.HBox([widgets.VBox([w_alter, w_wohnsitz, w_nationality]),
                  widgets.VBox([w_zivilstand, w_wohnform])]),
    widgets.HTML("<b>Education</b>"),
    widgets.HBox([widgets.VBox([w_ausb_typ, w_ausb_stufe, w_anerkannt]),
                  widgets.VBox([w_privat, w_qs, w_in_ausb])]),
    widgets.HBox([w_jahr, w_kum_jahre]),
    widgets.HTML("<b>Finances (applicant)</b>"),
    widgets.HBox([widgets.VBox([w_eink, w_verm, w_erwerb]),
                  widgets.VBox([w_ausb_kosten, w_kibe, w_fahrkosten_aus, w_mahlzeiten])]),
    widgets.HTML("<b>Eigener Haushalt / Ausnahmen</b>"),
    widgets.HBox([widgets.VBox([w_pendelzeit, w_zwingend, w_kinder_eigen, w_partnerschaft]),
                  widgets.VBox([w_vef, w_wiedereinstieg, w_wichtig, w_kum_darlehen])]),
])

In [6]:
# --- Parent A tab ---
pa_include = widgets.Checkbox(value=False, description="Include parent A", style=STYLE, layout=LAYOUT)
pa_alter = widgets.IntSlider(value=50, min=18, max=90, description="Alter", style=STYLE, layout=LAYOUT)
pa_eink = widgets.FloatText(value=0, description="Total Einkuenfte", style=STYLE, layout=LAYOUT)
pa_verm = widgets.FloatText(value=0, description="Vermoegen", style=STYLE, layout=LAYOUT)
pa_selbst = widgets.Checkbox(value=False, description="Selbststaendig", style=STYLE, layout=LAYOUT)
pa_ev = widgets.FloatText(value=0, description="Eigenmietwert", style=STYLE, layout=LAYOUT)
pa_unt = widgets.FloatText(value=0, description="Unterhalt geleistet", style=STYLE, layout=LAYOUT)
pa_steuern = widgets.FloatText(value=0, description="Bezahlte Steuern", style=STYLE, layout=LAYOUT)
pa_zahlt_unterhalt = widgets.Checkbox(value=False, description="Zahlt Unterhalt (Art.14 Abs.4)", style=STYLE, layout=LAYOUT)

parent_a_box = widgets.VBox([
    pa_include,
    widgets.HBox([widgets.VBox([pa_alter, pa_eink, pa_verm, pa_selbst]),
                  widgets.VBox([pa_ev, pa_unt, pa_steuern, pa_zahlt_unterhalt])]),
])

# --- Parent B tab ---
pb_include = widgets.Checkbox(value=False, description="Include parent B", style=STYLE, layout=LAYOUT)
pb_alter = widgets.IntSlider(value=48, min=18, max=90, description="Alter", style=STYLE, layout=LAYOUT)
pb_eink = widgets.FloatText(value=0, description="Total Einkuenfte", style=STYLE, layout=LAYOUT)
pb_verm = widgets.FloatText(value=0, description="Vermoegen", style=STYLE, layout=LAYOUT)
pb_selbst = widgets.Checkbox(value=False, description="Selbststaendig", style=STYLE, layout=LAYOUT)
pb_ev = widgets.FloatText(value=0, description="Eigenmietwert", style=STYLE, layout=LAYOUT)
pb_unt = widgets.FloatText(value=0, description="Unterhalt geleistet", style=STYLE, layout=LAYOUT)
pb_steuern = widgets.FloatText(value=0, description="Bezahlte Steuern", style=STYLE, layout=LAYOUT)
pb_zahlt_unterhalt = widgets.Checkbox(value=False, description="Zahlt Unterhalt (Art.14 Abs.4)", style=STYLE, layout=LAYOUT)

parent_b_box = widgets.VBox([
    pb_include,
    widgets.HBox([widgets.VBox([pb_alter, pb_eink, pb_verm, pb_selbst]),
                  widgets.VBox([pb_ev, pb_unt, pb_steuern, pb_zahlt_unterhalt])]),
])

In [7]:
# --- Partner tab ---
pt_include = widgets.Checkbox(value=False, description="Include partner", style=STYLE, layout=LAYOUT)
pt_alter = widgets.IntSlider(value=28, min=18, max=90, description="Alter", style=STYLE, layout=LAYOUT)
pt_eink = widgets.FloatText(value=0, description="Total Einkuenfte", style=STYLE, layout=LAYOUT)
pt_verm = widgets.FloatText(value=0, description="Vermoegen", style=STYLE, layout=LAYOUT)
pt_selbst = widgets.Checkbox(value=False, description="Selbststaendig", style=STYLE, layout=LAYOUT)

partner_box = widgets.VBox([
    widgets.HTML("<i>For Art. 32 Abs. 2 ABV (faktische Partnerschaft) qualification, also set the applicant's <code>zivilstand=faktische_partnerschaft</code> and <code>jahre_faktische_partnerschaft &gt;= 5</code> OR <code>hat_eigene_kinder=True</code>.</i>"),
    pt_include,
    widgets.HBox([widgets.VBox([pt_alter, pt_eink]),
                  widgets.VBox([pt_verm, pt_selbst])]),
])

# --- Household tab ---
hh_kinder = widgets.IntSlider(value=0, min=0, max=6, description="Kinder im Haushalt", style=STYLE, layout=LAYOUT)
hh_wk_familien = widgets.FloatText(value=14000, description="Wohnk. Familienb.", style=STYLE, layout=LAYOUT)
hh_wk_persoenlich = widgets.FloatText(value=11000, description="Wohnk. eig. Haushalt", style=STYLE, layout=LAYOUT)

household_box = widgets.VBox([
    widgets.HTML("<i>Children in the household are modelled with default age 10 (Krankenkasse Kinder-Bracket).</i>"),
    hh_kinder,
    widgets.HBox([hh_wk_familien, hh_wk_persoenlich]),
])

In [8]:
def collect_form():
    """Read every widget and produce kwargs for build_situation()."""
    applicant = {
        "alter": w_alter.value,
        "wohnsitz_grundlage": w_wohnsitz.value,
        "staatsangehoerigkeit": w_nationality.value,
        "zivilstand": w_zivilstand.value,
        "wohnform": w_wohnform.value,
        "ausbildungstyp": w_ausb_typ.value,
        "ausbildungsstufe": w_ausb_stufe.value,
        "ausbildungsstaette_anerkannt": w_anerkannt.value,
        "private_ausbildungsstaette": w_privat.value,
        "private_ausbildungsstaette_qualitaetsgesichert": w_qs.value,
        "in_ausbildung": w_in_ausb.value,
        "aktuelles_ausbildungsjahr": w_jahr.value,
        "kumulierte_ausbildungsjahre": w_kum_jahre.value,
        "total_einkuenfte_steuerveranlagung": w_eink.value,
        "steuerbares_vermoegen": w_verm.value,
        "erwerbseinkommen": w_erwerb.value,
        "tatsaechliche_ausbildungskosten": w_ausb_kosten.value,
        "kinderbetreuungskosten": w_kibe.value,
        "fahrkosten_ausbildung": w_fahrkosten_aus.value,
        "mahlzeiten_auswaerts_ausbildung_pro_jahr": w_mahlzeiten.value,
        "pendelzeit_zu_eltern_minuten": w_pendelzeit.value,
        "andere_zwingende_gruende_eigener_haushalt": w_zwingend.value,
        "hat_eigene_kinder": w_kinder_eigen.value,
        "jahre_faktische_partnerschaft": w_partnerschaft.value,
        "jahre_vollerwerbstaetigkeit": w_vef.value,
        "beruflicher_wiedereinstieg_nach_familienphase": w_wiedereinstieg.value,
        "wichtige_gruende_altersausnahme": w_wichtig.value,
        "kumulierte_darlehen": w_kum_darlehen.value,
    }
    parent_a = None
    if pa_include.value:
        parent_a = {
            "alter": pa_alter.value,
            "total_einkuenfte_steuerveranlagung": pa_eink.value,
            "steuerbares_vermoegen": pa_verm.value,
            "selbststaendig": pa_selbst.value,
            "eigenmietwert": pa_ev.value,
            "unterhaltsleistung_an_auszubildenden": pa_unt.value,
            "bezahlte_steuern": pa_steuern.value,
        }
    parent_b = None
    if pb_include.value:
        parent_b = {
            "alter": pb_alter.value,
            "total_einkuenfte_steuerveranlagung": pb_eink.value,
            "steuerbares_vermoegen": pb_verm.value,
            "selbststaendig": pb_selbst.value,
            "eigenmietwert": pb_ev.value,
            "unterhaltsleistung_an_auszubildenden": pb_unt.value,
            "bezahlte_steuern": pb_steuern.value,
        }
    partner = None
    if pt_include.value:
        partner = {
            "alter": pt_alter.value,
            "total_einkuenfte_steuerveranlagung": pt_eink.value,
            "steuerbares_vermoegen": pt_verm.value,
            "selbststaendig": pt_selbst.value,
        }
    household = {
        "wohnkosten_familienbudget_tatsaechlich": hh_wk_familien.value,
        "wohnkosten_persoenlich_tatsaechlich": hh_wk_persoenlich.value,
        "parent_a_zahlt_unterhalt": pa_zahlt_unterhalt.value if pa_include.value else False,
        "parent_b_zahlt_unterhalt": pb_zahlt_unterhalt.value if pb_include.value else False,
    }
    return dict(
        applicant=applicant,
        parent_a=parent_a,
        parent_b=parent_b,
        partner=partner,
        kinder_im_haushalt=hh_kinder.value,
        household=household,
    )

## 4. Compute & display

Pressing **Compute** assembles the situation, runs the simulation, and renders the eligibility / budget breakdown below the form.

In [14]:
compute_button = widgets.Button(description="Compute", button_style="primary")
show_situation_button = widgets.Button(description="Show raw situation")
out = widgets.Output()


def on_compute(_):
    with out:
        clear_output()
        try:
            kwargs = collect_form()
            situation = build_situation(**kwargs)
            results, _ = compute(situation)
            display(render_results(results))
        except Exception as exc:
            print(f"Error: {exc}")
            raise


def on_show(_):
    with out:
        clear_output()
        kwargs = collect_form()
        situation = build_situation(**kwargs)
        import json
        print(json.dumps(situation, indent=2, default=str))


compute_button.on_click(on_compute)
show_situation_button.on_click(on_show)

tabs = widgets.Tab(children=[applicant_box, parent_a_box, parent_b_box, partner_box, household_box])
for i, t in enumerate(["Applicant", "Parent A", "Parent B", "Partner", "Household"]):
    tabs.set_title(i, t)

form_ui = widgets.VBox([tabs, widgets.HBox([compute_button, show_situation_button]), out])
form_ui

## 5. Preset scenarios

These buttons load representative scenarios into the form. They mirror the cases in the YAML test suite at [`src/bern_stipendium/tests/`](../src/bern_stipendium/tests/).

Website results for presets: 
1. Sek-II apprentice, 18 years old, lives with parents: CHF 3950
2. Tertiary student, year 5: 2/3 stipendium + 1/3 darlehe: CHF 30 533
3. CHF 16 684
Stipendium: CHF 15,533.42  |  Darlehen: CHF 0  |  Stipendium-Anspruch: True
Eligibility (booleans)
hat_stipendienrechtlichen_wohnsitz_bern	True
erfuellt_persoenlichen_status	True
ausbildungstyp_anerkannt	True
ausbildungsstaette_qualifiziert	True
stipendium_grundsaetzlich_ausgeschlossen	False
verzicht_auf_anrechnung_eltern	True
beduerftig	True
maximale_beitragsdauer_eingehalten	True
altersgrenze_eingehalten	True
eigener_haushalt_anerkannt	True
partner_qualifiziert	True
stipendium_anspruch	True
darlehen_anspruch	True

Familienbudget (Art. 13-24 ABV)
haushaltsgroesse	1
familienbudget_einkommen	0
familienbudget_vermoegensanrechnung	0
familienbudget_grundbedarf	12,072
familienbudget_wohnkosten	0
familienbudget_krankenkasse	1,400
familienbudget_integrationszulage	0
familienbudget_einkommensfreibetrag	6,000
familienbudget_freibetraege_total	6,000
familienbudget_situationsbedingte_kosten	0
familienbudget_lebenshaltungskosten	13,472
familienbudget_saldo	-19,472

Persoenliches Budget (Art. 25-33 ABV)
persoenlich_personen_anzahl	2
persoenlich_einkommen	16,001
persoenlich_vermoegensanrechnung	0.15
persoenlich_grundbedarf	18,468
persoenlich_wohnkosten	11,000
persoenlich_krankenkasse	10,800
persoenlich_ausbildungskosten	3,000
persoenlich_situationsbedingte_kosten	3,800
familienbudget_ueberschuss_anteil_persoenlich	0
familienbudget_fehlbetrag_pro_kopf_anteil	19,472

Final amounts
anerkannte_ausbildungskosten	47,068
anrechenbare_mittel_total	16,001.15
fehlbetrag	15,533.42
stipendium_quote	1
stipendium_betrag	15,533.42
darlehen_betrag	0


In [10]:
def _set(widget, value):
    widget.value = value


def preset_lehrling(_):
    """Sek-II apprentice, 18 years old, lives with parents."""
    _set(w_alter, 18); _set(w_wohnsitz, "elterlicher_wohnsitz"); _set(w_nationality, "schweizer")
    _set(w_zivilstand, "ledig"); _set(w_wohnform, "elterlicher_haushalt")
    _set(w_ausb_typ, "erstausbildung"); _set(w_ausb_stufe, "sekundarstufe_ii"); _set(w_anerkannt, True)
    _set(w_jahr, 2); _set(w_kum_jahre, 1); _set(w_in_ausb, True)
    _set(w_eink, 2000); _set(w_verm, 0); _set(w_erwerb, 2000); _set(w_ausb_kosten, 1500)
    _set(w_mahlzeiten, 200); _set(w_fahrkosten_aus, 800)
    _set(w_pendelzeit, 0); _set(w_kum_darlehen, 0)
    _set(pa_include, True); _set(pa_alter, 50); _set(pa_eink, 30000); _set(pa_zahlt_unterhalt, False)
    _set(pb_include, True); _set(pb_alter, 48); _set(pb_eink, 25000); _set(pb_zahlt_unterhalt, False)
    _set(pt_include, False); _set(hh_kinder, 0)
    _set(hh_wk_familien, 14000); _set(hh_wk_persoenlich, 0)


def preset_tertiaer_jahr5(_):
    """Tertiary student, year 5: 2/3 stipendium + 1/3 darlehen."""
    _set(w_alter, 24); _set(w_wohnsitz, "finanziell_unabhaengig"); _set(w_nationality, "schweizer")
    _set(w_zivilstand, "ledig"); _set(w_wohnform, "eigener_haushalt")
    _set(w_ausb_typ, "erstausbildung"); _set(w_ausb_stufe, "tertiaerstufe"); _set(w_anerkannt, True)
    _set(w_jahr, 5); _set(w_kum_jahre, 4); _set(w_in_ausb, True)
    _set(w_eink, 6000); _set(w_verm, 0); _set(w_erwerb, 6000); _set(w_ausb_kosten, 4000)
    _set(w_mahlzeiten, 0); _set(w_fahrkosten_aus, 1200)
    _set(w_pendelzeit, 0); _set(w_kum_darlehen, 0)
    _set(pa_include, False); _set(pb_include, False); _set(pt_include, False); _set(hh_kinder, 0)
    _set(hh_wk_familien, 0); _set(hh_wk_persoenlich, 11000)


def preset_zweitausbildung(_):
    """Second training - Stipendium ausgeschlossen, nur Darlehen."""
    _set(w_alter, 28); _set(w_wohnsitz, "finanziell_unabhaengig"); _set(w_nationality, "schweizer")
    _set(w_zivilstand, "ledig"); _set(w_wohnform, "eigener_haushalt")
    _set(w_ausb_typ, "zweitausbildung"); _set(w_ausb_stufe, "tertiaerstufe"); _set(w_anerkannt, True)
    _set(w_jahr, 1); _set(w_kum_jahre, 0); _set(w_in_ausb, True)
    _set(w_eink, 1000); _set(w_verm, 0); _set(w_erwerb, 1000); _set(w_ausb_kosten, 3000)
    _set(w_kum_darlehen, 0)
    _set(pa_include, False); _set(pb_include, False); _set(pt_include, False); _set(hh_kinder, 0)
    _set(hh_wk_familien, 0); _set(hh_wk_persoenlich, 12000)


def preset_wiedereinstieg(_):
    """37yo career re-entry after family phase - age exception applies."""
    _set(w_alter, 37); _set(w_wohnsitz, "finanziell_unabhaengig"); _set(w_nationality, "schweizer")
    _set(w_zivilstand, "ledig"); _set(w_wohnform, "eigener_haushalt")
    _set(w_ausb_typ, "umschulung"); _set(w_ausb_stufe, "tertiaerstufe"); _set(w_anerkannt, True)
    _set(w_jahr, 1); _set(w_kum_jahre, 2); _set(w_in_ausb, True)
    _set(w_vef, 6); _set(w_wiedereinstieg, True)
    _set(w_eink, 8000); _set(w_verm, 0); _set(w_erwerb, 8000); _set(w_ausb_kosten, 3000)
    _set(w_kum_darlehen, 0)
    _set(pa_include, False); _set(pb_include, False); _set(pt_include, False); _set(hh_kinder, 0)
    _set(hh_wk_familien, 0); _set(hh_wk_persoenlich, 11000)


def preset_verheiratet(_):
    """Married applicant - Pro-Kopf-Anteil im persönlichen Budget (Art. 32)."""
    _set(w_alter, 28); _set(w_wohnsitz, "finanziell_unabhaengig"); _set(w_nationality, "schweizer")
    _set(w_zivilstand, "verheiratet"); _set(w_wohnform, "eigener_haushalt")
    _set(w_ausb_typ, "hoehere_berufsbildung"); _set(w_ausb_stufe, "tertiaerstufe"); _set(w_anerkannt, True)
    _set(w_jahr, 1); _set(w_kum_jahre, 0); _set(w_in_ausb, True)
    _set(w_eink, 5000); _set(w_verm, 0); _set(w_erwerb, 5000); _set(w_ausb_kosten, 3000)
    _set(w_kum_darlehen, 0)
    _set(pa_include, False); _set(pb_include, False)
    _set(pt_include, True); _set(pt_alter, 30); _set(pt_eink, 25000); _set(pt_verm, 0); _set(pt_selbst, False)
    _set(hh_kinder, 0); _set(hh_wk_familien, 0); _set(hh_wk_persoenlich, 14000)


preset_buttons = widgets.HBox([
    widgets.Button(description="Sek-II Lehrling 18"),
    widgets.Button(description="Tertiaer Jahr 5"),
    widgets.Button(description="Zweitausbildung"),
    widgets.Button(description="Wiedereinstieg 37"),
    widgets.Button(description="Verheiratet (Art. 32)"),
])
preset_buttons.children[0].on_click(preset_lehrling)
preset_buttons.children[1].on_click(preset_tertiaer_jahr5)
preset_buttons.children[2].on_click(preset_zweitausbildung)
preset_buttons.children[3].on_click(preset_wiedereinstieg)
preset_buttons.children[4].on_click(preset_verheiratet)
preset_buttons

## 6. Sensitivity sweep

Pick one applicant input and watch the resulting `stipendium_betrag` and `darlehen_betrag` change as it varies. Useful for spotting the 2/3-quote breakpoint at tertiary year 4 or the lifetime-loan cap.

In [ ]:
SWEEPABLE = [
    ("alter", 15, 65, 1),
    ("aktuelles_ausbildungsjahr", 1, 12, 1),
    ("kumulierte_ausbildungsjahre", 0, 15, 1),
    ("total_einkuenfte_steuerveranlagung", 0, 60000, 2000),
    ("steuerbares_vermoegen", 0, 200000, 10000),
    ("erwerbseinkommen", 0, 30000, 1000),
    ("tatsaechliche_ausbildungskosten", 0, 8000, 500),
    ("kumulierte_darlehen", 0, 60000, 2000),
    ("jahre_vollerwerbstaetigkeit", 0, 10, 1),
]

sweep_field = widgets.Dropdown(
    options=[(f"{n} ({lo}..{hi}, step {st})", (n, lo, hi, st)) for (n, lo, hi, st) in SWEEPABLE],
    description="Sweep over", style=STYLE, layout=widgets.Layout(width="540px"),
)
sweep_button = widgets.Button(description="Run sweep", button_style="info")
sweep_out = widgets.Output()


def on_sweep(_):
    with sweep_out:
        clear_output()
        name, lo, hi, step = sweep_field.value
        kwargs = collect_form()
        rows = []
        for v in range(lo, hi + 1, step):
            kwargs["applicant"][name] = v
            situation = build_situation(**kwargs)
            results, _ = compute(situation)
            rows.append({
                name: v,
                "fehlbetrag": float(results["fehlbetrag"]),
                "stipendium_betrag": float(results["stipendium_betrag"]),
                "darlehen_betrag": float(results["darlehen_betrag"]),
                "stipendium_anspruch": bool(results["stipendium_anspruch"]),
            })
        df = pd.DataFrame(rows)
        display(df)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(df[name], df["stipendium_betrag"], marker="o", label="Stipendium")
        ax.plot(df[name], df["darlehen_betrag"], marker="s", label="Darlehen")
        ax.plot(df[name], df["fehlbetrag"], linestyle="--", alpha=0.5, label="Fehlbetrag")
        ax.set_xlabel(name)
        ax.set_ylabel("CHF")
        ax.legend(); ax.grid(True)
        ax.set_title(f"Sensitivity to {name}")
        plt.show()


sweep_button.on_click(on_sweep)
widgets.VBox([widgets.HBox([sweep_field, sweep_button]), sweep_out])